In [1]:
!pip install transformers datasets scikit-learn

import json
import re
import torch
import numpy as np
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

# =========================
# 1. Load Dataset
# =========================
with open("/content/CN_relations_fixed.json") as f:
    data = json.load(f)

# =========================
# 2. Extract Text + Labels
# =========================
texts = []
labels = []

for d in data:
    text = d["input"]
    label = d["output"].strip()
    texts.append(text)
    labels.append(label)

# Define label set
label_list = ["used_for", "based_on", "implements", "part_of", "improves", "no_relation"]

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

# Handle unknown labels
labels = [l if l in label2id else "no_relation" for l in labels]

# =========================
# 3. Train-Test Split
# =========================
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

train_dataset = Dataset.from_dict({
    "text": train_texts,
    "label": [label2id[l] for l in train_labels]
})

val_dataset = Dataset.from_dict({
    "text": val_texts,
    "label": [label2id[l] for l in val_labels]
})

# =========================
# 4. Tokenization
# =========================
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=256)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# =========================
# 5. Model
# =========================
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# =========================
# 6. Metrics
# =========================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(
        labels, preds, average="micro", zero_division=0
    )

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision_micro": precision_micro,
        "recall_micro": recall_micro,
        "f1_micro": f1_micro,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro
    }

# =========================
# 7. Training Arguments
# =========================
training_args = TrainingArguments(
    output_dir="./roberta-relation",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    learning_rate=2e-5,
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro"   # ✅ fixed key
)

# =========================
# 8. Trainer
# =========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# =========================
# 9. Train
# =========================
trainer.train()

# =========================
# 10. FINAL EVALUATION + CLEAN PRINT
# =========================
metrics = trainer.evaluate()

print("\n" + "="*40)
print("📊 FINAL EVALUATION RESULTS")
print("="*40)

print(f"Accuracy            : {metrics['eval_accuracy']:.4f}")
print(f"Precision (Micro)   : {metrics['eval_precision_micro']:.4f}")
print(f"Recall (Micro)      : {metrics['eval_recall_micro']:.4f}")
print(f"F1 Score (Micro)    : {metrics['eval_f1_micro']:.4f}")
print(f"Precision (Macro)   : {metrics['eval_precision_macro']:.4f}")
print(f"Recall (Macro)      : {metrics['eval_recall_macro']:.4f}")
print(f"F1 Score (Macro)    : {metrics['eval_f1_macro']:.4f}")

print("="*40)

# =========================
# 11. OPTIONAL: CLASSIFICATION REPORT
# =========================
predictions = trainer.predict(val_dataset)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

print("\n📋 Detailed Classification Report:\n")
print(classification_report(labels, preds, target_names=label_list, zero_division=0))

# =========================
# 12. Entity Extraction Function
# =========================
def extract_entities(text):
    e1 = re.search(r'Entity1:\s*(.*?)\s*\(', text)
    e2 = re.search(r'Entity2:\s*(.*?)\s*\(', text)

    return (
        e1.group(1).strip() if e1 else "",
        e2.group(1).strip() if e2 else ""
    )

# =========================
# 13. Prediction + Triplets
# =========================
model.eval()

triplets = []

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

for text in val_texts:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        pred = torch.argmax(outputs.logits, dim=1).item()

    relation = id2label[pred]
    e1, e2 = extract_entities(text)

    triplets.append({
        "entity1": e1,
        "relation": relation,
        "entity2": e2
    })

# =========================
# 14. Save Triplets
# =========================
with open("predicted_triplets.json", "w") as f:
    json.dump(triplets, f, indent=4)

print("\n✅ Triplets saved to predicted_triplets.json")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


📊 FINAL EVALUATION RESULTS
Accuracy            : 0.8414
Precision (Micro)   : 0.8414
Recall (Micro)      : 0.8414
F1 Score (Micro)    : 0.8414
Precision (Macro)   : 0.8242
Recall (Macro)      : 0.6596
F1 Score (Macro)    : 0.7044

📋 Detailed Classification Report:

              precision    recall  f1-score   support

    used_for       0.82      0.90      0.86       132
    based_on       1.00      0.62      0.76        13
  implements       0.67      0.14      0.24        14
     part_of       0.79      0.73      0.76        37
    improves       0.80      0.67      0.73        12
 no_relation       0.86      0.90      0.88       183

    accuracy                           0.84       391
   macro avg       0.82      0.66      0.70       391
weighted avg       0.84      0.84      0.83       391


✅ Triplets saved to predicted_triplets.json
